<img src='https://hammondm.github.io/hltlogo1.png' style="float:right">

LING 593A-011<br>
Fall 2025<br>
Davo Acevedo-Cardona

# GUU — Firebase Upload Tool

This notebook is a test sample to upload all the CSV to firebase.

Sai, Lindsey and I still need to agree where the data will be uploaded (Firebase project).


# Imports

In [1]:
# Cell 1 — Imports + load input (brand_request_counts.csv)
import re
import pandas as pd

IN_COUNTS = "brand_request_counts.csv"

df = pd.read_csv(IN_COUNTS)
# expected columns: brand_canonical, count
df["brand_canonical"] = df["brand_canonical"].astype(str)
df["count"] = pd.to_numeric(df["count"], errors="coerce").fillna(0).astype(int)

print("Loaded:", df.shape)
df.head()


Loaded: (35286, 4)


,brand_canonical,count,first_seen,last_seen
0,Subway,2683,2017-11-24T22:49:31+00:00,2024-12-31T15:16:56-08:00
1,Out,1403,2017-11-24T16:18:39-05:00,2024-12-31T03:05:38+00:00
2,Publix,1104,2018-07-07T17:55:38-04:00,2024-12-31T12:35:58-05:00
3,Sprouts,1044,2018-01-13T15:44:29-06:00,2024-12-31T09:12:54-07:00
4,Home Depot,906,2018-07-22T20:19:37-05:00,2024-12-30T11:21:52-06:00


# Cleaning rules

In [2]:
# Cell 2 — Normalization + trimming rules (remove commentary tails)
STATE_HINT = r"(alabama|alaska|arizona|arkansas|california|colorado|connecticut|delaware|florida|georgia|hawaii|idaho|illinois|indiana|iowa|kansas|kentucky|louisiana|maine|maryland|massachusetts|michigan|minnesota|mississippi|missouri|montana|nebraska|nevada|new hampshire|new jersey|new mexico|new york|north carolina|north dakota|ohio|oklahoma|oregon|pennsylvania|rhode island|south carolina|south dakota|tennessee|texas|utah|vermont|virginia|washington|west virginia|wisconsin|wyoming)"

TRIM_PATTERNS = [
    r"\s*(>\s*>)+\s*",                                 # reply artifacts >> >>
    r"\b(sent from|thanks|thank you)\b.*$",             # email footer-ish
    r"\bdo they\b.*$",                                 # questions
    r"\b(does it|is it)\b.*$",                          # questions
    r"\b(support|democrat|republican|progressive)\b.*$",# political commentary tail
    r"\bi think\b.*$|\bi assume\b.*$|\bim assuming\b.*$",
    r"\bwebsite\b.*$|\bwww\..*$|\bhttps?://.*$",
    r"\bbased in\b.*$|\blocated\b.*$",
    r"\bin\s+" + STATE_HINT + r"\b.*$",                 # "in Ohio ..."
    r"\bnear me\b.*$",
    r"\bthey are\b.*$|\bparent company\b.*$|\band parent company\b.*$",
    r"\baka\b.*$|\ba\.k\.a\.\b.*$",
    r"\bwhere you can\b.*$",
    r"\b(an?|the)\s+parent\s+company\b.*$",
]

def fix_mojibake(s: str) -> str:
    if not isinstance(s, str):
        return ""
    return (s
        .replace("â€™", "’")
        .replace("â€˜", "‘")
        .replace("â€œ", "“")
        .replace("â€\u009d", "”")
        .replace("â€“", "–")
        .replace("â€”", "—")
        .replace("Ã©", "é")
        .replace("Ã¨", "è")
        .replace("Ã¡", "á")
        .replace("Ã³", "ó")
        .replace("Ã±", "ñ")
        .replace("Ã¼", "ü")
    )

def normalize_text(s: str) -> str:
    s = fix_mojibake(s)
    s = re.sub(r"\s+", " ", s).strip()
    s = s.strip(" -–—|•\t\r\n")
    return s

def trim_commentary(s: str) -> str:
    out = normalize_text(s)
    for pat in TRIM_PATTERNS:
        new = re.sub(pat, " ", out, flags=re.I).strip()
        # if a pattern changed it, keep the change
        if new != out:
            out = re.sub(r"\s+", " ", new).strip()
    # keep left of ":" (often "Brand: message")
    if ":" in out:
        out = out.split(":", 1)[0].strip()
    # keep left of "(" if it looks like commentary
    if "(" in out and len(out) > 12:
        out = out.split("(", 1)[0].strip()
    return out

# Apply cleaning + re-aggregate counts

In [3]:
# Cell 3 — Alias + canonical fixes (add to this over time)
ALIAS_MAP = {
    # common typos seen in your outputs
    "Cosco": "Costco",
    "Khols": "Kohls",
    "Humanna": "Humana",
    "Safeways": "Safeway",
    "Sketchers": "Skechers",
    "Sketcher": "Skechers",
    "Jersy Mikes": "Jersey Mike's",
    "Dominoes": "Domino's",
    "Mcdonalds": "McDonald's",
    "Anytimefitness": "Anytime Fitness",
    "Hungry Root": "Hungryroot",
    "Hyvee": "Hy-Vee",
    "Shoprite": "Shop Rite",
    "Tommy'S Car Wash": "Tommy's Car Wash",
}

def apply_alias(s: str) -> str:
    # normalize apostrophes before mapping
    s = s.replace("’", "'")
    return ALIAS_MAP.get(s, s)


# Save outputs

In [4]:
import re

URL_RE = re.compile(r"(https?://|www\.|\.(com|net|org|io|co)\b)", re.I)
EMAIL_RE = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b", re.I)

# Allow normal brand punctuation
ALLOWED_CHARS_RE = re.compile(r"^[A-Za-z0-9&'’\-\.\s]+$")

# Only phrases that are *actually* commentary/meta text (high precision)
COMMENTARY_RE = re.compile(
    r"\b("
    r"do they|does it|support|democrat|democrats|republican|republicans|progressive|"
    r"i think|i assume|im assuming|i'm assuming|"
    r"parent company|and parent company|they are the parent company|"
    r"website is|website|sent from|thanks|thank you|"
    r"where you can|based in|located|near me"
    r")\b",
    re.I,
)

DROP_EXACT = {
    "", "Stop", "Out", "Wings", "Cooling", "Etc",
    "A Meal Delivery Company",
    "Cell Phone Services Provider",
    "Fast Foods & Burgers Drive-Thru",
    "They Are The Parent Company",
}

def looks_like_brand_v2(s: str) -> tuple[str, str]:
    """
    Returns (label, reason)
    label: keep / review / drop
    """
    s = (s or "").strip()
    if not s:
        return "drop", "empty"

    # obvious junk
    if s in DROP_EXACT:
        return "drop", "drop_exact"

    if URL_RE.search(s) or EMAIL_RE.search(s):
        return "drop", "url_or_email"

    if not ALLOWED_CHARS_RE.match(s):
        return "drop", "weird_chars"

    # commentary present => usually still salvageable by trimming,
    # but if it remains after trimming, send to review not drop
    if COMMENTARY_RE.search(s):
        # If it's very long, treat as drop; otherwise review
        if len(s) > 60 or len(s.split()) > 10:
            return "drop", "commentary_long"
        return "review", "commentary"

    # still too long = likely sentence
    if len(s) > 55 or len(s.split()) > 8:
        return "review", "too_long"

    # otherwise, keep
    return "keep", "ok"


In [5]:
import pandas as pd
import re

work = df.copy()

work["candidate"] = (
    work["brand_canonical"]
      .map(trim_commentary)
      .map(normalize_text)
      .map(apply_alias)
      .map(lambda x: re.sub(r"\s+", " ", x).strip())
)

labels = work["candidate"].map(looks_like_brand_v2)
work["label"] = labels.map(lambda t: t[0])
work["reason"] = labels.map(lambda t: t[1])

kept = work[work["label"] == "keep"].copy()
review = work[work["label"] == "review"].copy()
dropped = work[work["label"] == "drop"].copy()

clean_counts = (
    kept
    .groupby("candidate", as_index=False)
    .agg(
        count=("count", "sum"),
        first_seen=("first_seen", "min"),
        last_seen=("last_seen", "max"),
    )
    .sort_values("count", ascending=False)
    .rename(columns={"candidate": "brand_canonical_clean"})
)


clean_counts.to_csv("brand_request_counts_clean.csv", index=False)
review[["brand_canonical","candidate","count","reason"]]\
    .sort_values("count", ascending=False)\
    .to_csv("brand_request_counts_review.csv", index=False)
dropped[["brand_canonical","candidate","count","reason"]]\
    .sort_values("count", ascending=False)\
    .to_csv("brand_request_counts_dropped.csv", index=False)

print("Input rows:", len(df))
print("Kept rows:", len(kept), "=> unique kept:", len(clean_counts))
print("Review rows:", len(review))
print("Dropped rows:", len(dropped))

clean_counts.head(40)


Input rows: 35286
Kept rows: 34876 => unique kept: 34852
Review rows: 19
Dropped rows: 391


,brand_canonical_clean,count,first_seen,last_seen
29255,Subway,2683,2017-11-24T22:49:31+00:00,2024-12-31T15:16:56-08:00
24776,Publix,1104,2018-07-07T17:55:38-04:00,2024-12-31T12:35:58-05:00
28651,Sprouts,1044,2018-01-13T15:44:29-06:00,2024-12-31T09:12:54-07:00
14253,Home Depot,906,2018-07-22T20:19:37-05:00,2024-12-30T11:21:52-06:00
14966,In-N-Out,697,2018-08-30T11:58:02+03:00,2024-12-31T14:44:42-07:00
33295,Walmart,654,2018-09-19T15:20:36-05:00,2024-12-30T11:53:38-08:00
4466,Brandy Melville,616,2018-08-28T13:31:49-07:00,2024-12-17T18:00:47-05:00
29240,Subaru,600,2017-12-09T23:21:34-05:00,2024-12-30T15:40:36-06:00
25200,Raising Canes,577,2018-04-15T14:43:55-05:00,2024-12-31T09:33:06-07:00
24807,Puma,560,2017-12-13T19:28:22-08:00,2024-12-31T09:14:59-06:00


In [6]:
# Cell 6 — Inspect top REVIEW (v2 columns)
print("Top REVIEW:")
review.sort_values("count", ascending=False).head(60)[
    ["brand_canonical", "candidate", "count", "reason"]
]


Top REVIEW:


,brand_canonical,candidate,count,reason
1569,Spiritual Gangster Clothing Company Aviator Na...,Spiritual Gangster Clothing Company Aviator Na...,19,too_long
2709,Tru Earth Maybe Spelled Truearth Dissolvable L...,Tru Earth Maybe Spelled Truearth Dissolvable L...,11,too_long
3198,Republicans,Republicans,10,commentary
5377,Versa Capital Management Owners Of Black Angus...,Versa Capital Management Owners Of Black Angus...,5,too_long
5989,Or Other Women'S Undergarments Companies Besid...,Or Other Women'S Undergarments Companies Besid...,5,too_long
6002,Wesco Gas Stations Headquarters In North Muske...,Wesco Gas Stations Headquarters In North Muske...,5,too_long
7204,Canton School Employees Federal Credit Union C...,Canton School Employees Federal Credit Union C...,4,too_long
7852,Washington Commanders. You Still List The Wash...,Washington Commanders. You Still List The Wash...,3,too_long
10888,Mz Industries Originally Massimo Zanetti Bever...,Mz Industries Originally Massimo Zanetti Bever...,2,too_long
13602,Guaranteed Rate - Mortgage Company Headquarter...,Guaranteed Rate - Mortgage Company Headquarter...,2,too_long


In [7]:
# Cell 7 — Inspect top DROPPED (v2 columns)
print("Top DROPPED:")
dropped.sort_values("count", ascending=False).head(60)[
    ["brand_canonical", "candidate", "count", "reason"]
]


Top DROPPED:


,brand_canonical,candidate,count,reason
1,Out,Out,1403,drop_exact
36,Stop,Stop,343,drop_exact
238,Progressive Insurance,,101,empty
337,Wings,Wings,75,drop_exact
539,Etc,Etc,50,drop_exact
902,Batteries +,Batteries +,32,weird_chars
960,Rodan + Fields,Rodan + Fields,31,weird_chars
1535,"Ollie'S ""Good Stuff Cheap""","Ollie'S ""Good Stuff Cheap""",20,weird_chars
2132,Häagen-Dazs,Häagen-Dazs,14,weird_chars
2339,Fila <Https,Fila <Https,13,weird_chars
